# Predicting next-month account churn

This notebook tests whether the available monthly snapshot data can rank accounts by the probability of cancelling in the following month. It emphasizes a deployment-like time split and explicitly removes a target-leaking field.

**Bottom line:** yes, there is useful but moderate predictive signal. On the untouched November 2025 snapshots, the selected leakage-free model achieves about **0.71 ROC AUC** and **0.30 average precision**, compared with an **11.3%** churn rate. The highest-scored 10% of accounts contain about **35% churners** (roughly 3.1× lift) and capture about **31% of all churners**. This supports prioritization, not automatic decisions about individual customers.

## 1. Setup and data validation

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
pd.set_option("display.max_columns", 50)

data_path = Path("data.csv")
dictionary_path = Path("columns.csv")
assert data_path.exists() and dictionary_path.exists(), "Run from the folder containing both CSV files."

data = pd.read_csv(data_path)
dictionary = pd.read_csv(dictionary_path)
data["snapshot_month"] = pd.to_datetime(data["snapshot_month"], format="%Y-%m")

expected = set(dictionary["column"])
assert expected == set(data.columns), "Data columns differ from the dictionary."
assert data["churn_next_month"].isin([0, 1]).all()
assert data.isna().sum().sum() == 0

print(f"Rows: {len(data):,}; accounts: {data['account_ref'].nunique():,}")
print(f"Months: {data['snapshot_month'].min():%Y-%m} through {data['snapshot_month'].max():%Y-%m}")
display(dictionary)

There are repeated snapshots per account, which is expected for this panel dataset. A random row split would mix past and future periods and give an optimistic estimate. We therefore reserve the latest month as a final test set.

In [ ]:
monthly = (
    data.groupby("snapshot_month")["churn_next_month"]
    .agg(rows="size", churners="sum", churn_rate="mean")
)
display(monthly.style.format({"churn_rate": "{:.1%}"}))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly.index, monthly["churn_rate"], marker="o")
ax.axhline(data["churn_next_month"].mean(), color="0.5", ls="--", label="overall rate")
ax.set(title="Churn rate changes substantially over time", ylabel="Next-month churn rate", xlabel="Snapshot month")
ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Leakage audit

`lifetime_months` is described as total lifetime from a CRM export dated 2026-01-05. That information would not have existed at each historical scoring date. In fact, `lifetime_months == tenure_months` identifies every positive label exactly, while every non-churn row has additional future lifetime. It is therefore direct future leakage and must not be used. `account_ref` is also excluded because it is an identifier rather than a generalizable account attribute.

In [ ]:
leak_rule = (data["lifetime_months"] == data["tenure_months"]).astype(int)
leak_check = pd.crosstab(
    leak_rule.rename("lifetime equals tenure"),
    data["churn_next_month"],
    margins=True,
)
display(leak_check)
print(f"Accuracy of this impossible-at-score-time rule: {(leak_rule == data['churn_next_month']).mean():.1%}")

## 3. Time-based model selection

October 2025 is used only to select a training-history window; November 2025 remains untouched until the final evaluation. Because the churn rate shifted sharply over the data history, we compare expanding history with trailing 6-, 9-, and 12-month windows.

The model is regularized logistic regression: transparent, stable on this small tabular dataset, and able to produce probabilities. `snapshot_month` itself is excluded because calendar time is not a causal account attribute and extrapolating it after the historical churn spike produced unstable probabilities.

In [ ]:
target = "churn_next_month"
categorical_features = ["plan"]
numeric_features = ["seats", "monthly_fee", "tenure_months", "usage_hours", "support_tickets"]
features = categorical_features + numeric_features

def make_model():
    preprocessing = ColumnTransformer([
        ("plan", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", StandardScaler(), numeric_features),
    ])
    return Pipeline([
        ("preprocessing", preprocessing),
        ("model", LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE)),
    ])

def probability_metrics(y_true, probability):
    return {
        "roc_auc": roc_auc_score(y_true, probability),
        "average_precision": average_precision_score(y_true, probability),
        "brier_score": brier_score_loss(y_true, probability),
        "mean_prediction": np.mean(probability),
    }

validation_month = pd.Timestamp("2025-10-01")
test_month = pd.Timestamp("2025-11-01")
validation_mask = data["snapshot_month"].eq(validation_month)
test_mask = data["snapshot_month"].eq(test_month)

print("Validation:", validation_mask.sum(), "rows,", f"{data.loc[validation_mask, target].mean():.1%} churn")
print("Test:      ", test_mask.sum(), "rows,", f"{data.loc[test_mask, target].mean():.1%} churn")

In [ ]:
candidate_windows = {"expanding": None, "trailing_6m": 6, "trailing_9m": 9, "trailing_12m": 12}
selection_rows = []

for name, months in candidate_windows.items():
    train_mask = data["snapshot_month"].lt(validation_month)
    if months is not None:
        first_month = validation_month - pd.DateOffset(months=months)
        train_mask &= data["snapshot_month"].ge(first_month)

    candidate = make_model()
    candidate.fit(data.loc[train_mask, features], data.loc[train_mask, target])
    validation_probability = candidate.predict_proba(data.loc[validation_mask, features])[:, 1]
    selection_rows.append({
        "window": name,
        "training_rows": int(train_mask.sum()),
        **probability_metrics(data.loc[validation_mask, target], validation_probability),
    })

selection = pd.DataFrame(selection_rows).set_index("window").sort_values("average_precision", ascending=False)
display(selection.style.format({
    "roc_auc": "{:.3f}", "average_precision": "{:.3f}",
    "brier_score": "{:.3f}", "mean_prediction": "{:.1%}",
}))

selected_name = selection.index[0]
selected_months = candidate_windows[selected_name]
print("Selected by validation average precision:", selected_name)

Average precision is the primary selection metric because churn is uncommon and the practical use case is ranking accounts for outreach. ROC AUC measures overall discrimination; the Brier score measures probability error (lower is better).

## 4. One-time evaluation on the untouched latest month

After choosing the window on October, the same specification is refit through October and evaluated once on November. This mirrors a monthly retraining workflow.

In [ ]:
final_train_mask = data["snapshot_month"].lt(test_month)
if selected_months is not None:
    final_first_month = test_month - pd.DateOffset(months=selected_months)
    final_train_mask &= data["snapshot_month"].ge(final_first_month)

final_model = make_model()
final_model.fit(data.loc[final_train_mask, features], data.loc[final_train_mask, target])
test_y = data.loc[test_mask, target].to_numpy()
test_probability = final_model.predict_proba(data.loc[test_mask, features])[:, 1]

prior_probability = np.repeat(data.loc[final_train_mask, target].mean(), len(test_y))
test_results = pd.DataFrame([
    {"model": "constant prior baseline", **probability_metrics(test_y, prior_probability)},
    {"model": f"logistic ({selected_name})", **probability_metrics(test_y, test_probability)},
]).set_index("model")
display(test_results.style.format({
    "roc_auc": "{:.3f}", "average_precision": "{:.3f}",
    "brier_score": "{:.3f}", "mean_prediction": "{:.1%}",
}))
print(f"Actual November churn rate: {test_y.mean():.1%}")

In [ ]:
def targeting_at_fraction(y_true, probability, fraction):
    count = int(np.ceil(len(y_true) * fraction))
    selected = np.argsort(-probability)[:count]
    precision = y_true[selected].mean()
    recall = y_true[selected].sum() / y_true.sum()
    return {
        "targeted_fraction": fraction, "accounts_targeted": count,
        "precision": precision, "recall": recall, "lift": precision / y_true.mean(),
    }

targeting = pd.DataFrame([
    targeting_at_fraction(test_y, test_probability, fraction)
    for fraction in (0.10, 0.20, 0.30)
]).set_index("targeted_fraction")
display(targeting.style.format({"precision": "{:.1%}", "recall": "{:.1%}", "lift": "{:.2f}×"}))

The targeting table is often easier to act on than a universal 0.5 cutoff. For example, if the retention team can contact 10% of accounts, it can use the top decile of scores and expect materially higher churn concentration than random selection. The operating cutoff should ultimately be chosen from outreach capacity, intervention cost, and expected retained value.

## 5. Uncertainty, calibration, and interpretation

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
bootstrap_scores = []
for _ in range(2_000):
    sampled = rng.integers(0, len(test_y), len(test_y))
    if np.unique(test_y[sampled]).size < 2:
        continue
    bootstrap_scores.append([
        roc_auc_score(test_y[sampled], test_probability[sampled]),
        average_precision_score(test_y[sampled], test_probability[sampled]),
    ])

intervals = pd.DataFrame(
    np.percentile(bootstrap_scores, [2.5, 97.5], axis=0).T,
    index=["ROC AUC", "Average precision"], columns=["2.5%", "97.5%"],
)
display(intervals.style.format("{:.3f}"))

In [ ]:
calibration = pd.DataFrame({"actual": test_y, "predicted": test_probability})
calibration["risk_bin"] = pd.qcut(calibration["predicted"], q=10, duplicates="drop")
calibration_table = calibration.groupby("risk_bin", observed=True).agg(
    accounts=("actual", "size"), mean_prediction=("predicted", "mean"), actual_rate=("actual", "mean")
).reset_index(drop=True)
display(calibration_table.style.format({"mean_prediction": "{:.1%}", "actual_rate": "{:.1%}"}))

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 0.5], [0, 0.5], ls="--", color="0.6", label="perfect calibration")
ax.plot(calibration_table["mean_prediction"], calibration_table["actual_rate"], marker="o", label="risk deciles")
ax.set(xlabel="Mean predicted probability", ylabel="Observed churn rate", title="November calibration by score decile")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
feature_names = final_model.named_steps["preprocessing"].get_feature_names_out()
coefficients = pd.Series(
    final_model.named_steps["model"].coef_[0], index=feature_names, name="log_odds_coefficient"
).sort_values()
display(coefficients.to_frame().style.format("{:+.3f}"))

Numeric coefficients are per one standard deviation because those variables were standardized. They describe associations conditional on the other fields, not causal effects. More support tickets are the strongest positive risk signal; more usage is the strongest negative signal. Correlated commercial fields such as plan, seats, and fee should not be interpreted independently as causal.

## Conclusion and next steps

The data supports a useful **risk-ranking model**. The final out-of-time result is materially better than the base rate, but its bootstrap interval also shows meaningful uncertainty, and discrimination is not strong enough to treat scores as certain individual outcomes.

Before production use:

1. Remove `lifetime_months` from the scoring feature contract, not merely from this notebook.
2. Refit monthly on a recent rolling window and monitor churn rate, average precision, calibration, and top-decile lift.
3. Backfill richer pre-snapshot signals—usage trend, days since last activity, payment failures, ticket sentiment/severity, and contract/renewal timing.
4. Choose the contact threshold from campaign capacity and economics, then measure whether intervention reduces churn with a randomized holdout. Prediction alone does not establish that outreach causes retention.
5. Confirm that the most recent labels have completed their full next-month observation window before retraining.